# 2.1 Converting MCQ to OSQ

# Define Prompts

In [11]:
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"}
    ],
    temperature=0.7  # <--- add temperature here
)

print(completion.choices[0].message.content)


Hello! How can I assist you today?


In [12]:
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain the importance of systems engineering."}
    ],
    temperature=0.7  # <--- add temperature here
)

print(completion.choices[0].message.content)


Systems engineering is a crucial discipline for several reasons:

1. Integration: Systems engineering ensures different components of a project or system work together seamlessly. It focuses on designing and managing complex systems over their life cycles ensuring that all different parts are integrated properly.

2. Problem-solving: Systems engineering provides a structured solution process that considers both the business and technical needs of all customers. This holistic view enables engineers to create a solution that not only solves the problem at hand but also works in the larger system.

3. Risk Management: It allows for early detection of possible risks and issues, enabling preventative action and ensuring that the system can meet its requirements even in unforeseen circumstances.

4. Cost Effective: By considering the entire life cycle of a system, from conception to decommissioning, systems engineering helps to reduce costs and prevent overruns in both time and resources.

5

# Prompts

In [ ]:
# -----------------------------
# Helper: call API and parse JSON
# -----------------------------
def chat_json(prompt: str,
              model: str = "gpt-4o-mini",
              temperature: float = 0.0,
              max_tokens: int = 600) -> dict:
    """Call Chat Completions with low temperature and parse JSON."""
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    text = resp.choices[0].message.content.strip()
    try:
        if text.startswith("```"):
            text = text.strip("`")
            text = text[text.find("{"):]
        return json.loads(text)
    except Exception:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end+1])
        raise ValueError(f"Model did not return valid JSON:\n{text}")

# -----------------------------
# Step 1: classify suitability
# -----------------------------
def classify_mcq_suitability(df: pd.DataFrame,
                             question_col: str = "question",
                             model: str = "gpt-4o-mini") -> pd.DataFrame:
    decisions, reasons = [], []
    for q in df[question_col].tolist():
        prompt = f"{CLASSIFIER_PROMPT}\n\nMCQ:\n{q}"
        out = chat_json(prompt, model=model, temperature=0.0, max_tokens=200)
        decisions.append(out.get("decision", "No"))
        reasons.append(out.get("reason", ""))
    df = df.copy()
    df["osq_suitable"] = decisions
    df["osq_suitable_reason"] = reasons
    return df

# -----------------------------
# Step 2: convert MCQ → OSQ
# -----------------------------
def convert_mcq_row(row: pd.Series,
                    model: str = "gpt-4o-mini",
                    temperature: float = 0.2,
                    max_tokens: int = 800) -> dict:
    q = row.get("question", "")
    choices = row.get("choices", [])
    correct_choice = row.get("correct_choice", "")
    explanation = row.get("explanation", "") or "N/A"

    prompt = (
        f"{CONVERTER_HEADER}\n\n"
        f"Input MCQ:\n"
        f"- Stem: {q}\n"
        f"- Choices: {json.dumps(choices, ensure_ascii=False)}\n"
        f"- Correct choice: {correct_choice}\n"
        f"- Reference explanation (if any): {explanation}\n"
    )
    return chat_json(prompt, model=model, temperature=temperature, max_tokens=max_tokens)

def convert_mcqs_to_osq(df: pd.DataFrame,
                        model: str = "gpt-4o-mini",
                        only_if_suitable: bool = True) -> pd.DataFrame:
    df = df.copy()
    osq_prompt, expected, rubric, difficulty = [], [], [], []
    for _, row in df.iterrows():
        if only_if_suitable and str(row.get("osq_suitable", "No")) != "Yes":
            osq_prompt.append(None)
            expected.append(None)
            rubric.append(None)
            difficulty.append(None)
            continue
        out = convert_mcq_row(row, model=model)
        osq_prompt.append(out.get("osq_prompt"))
        expected.append(out.get("expected_answer"))
        rubric.append(out.get("rubric"))
        difficulty.append(out.get("difficulty"))
    df["osq_prompt"] = osq_prompt
    df["expected_answer"] = expected
    df["rubric"] = rubric
    df["difficulty"] = difficulty
    return df

# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    mcqs = pd.DataFrame([
        {
            "id": "Q1",
            "question": "Which lifecycle model best supports iterative risk reduction through continuous stakeholder feedback?",
            "choices": ["Waterfall", "Spiral", "V-Model", "Big-Bang"],
            "correct_choice": "Spiral",
            "explanation": "The Spiral model emphasizes iterative cycles with risk-driven prototyping and stakeholder input."
        },
        {
            "id": "Q2",
            "question": "What is the formula for Shannon capacity of an AWGN channel?",
            "choices": ["C=B log2(1+SNR)", "C=B ln(1+SNR)", "C=B log10(1+SNR)", "C=log2(1+SNR)"],
            "correct_choice": "C=B log2(1+SNR)",
            "explanation": "Capacity is C = B * log2(1 + SNR)."
        }
    ])

    # Step 1: classify
    mcqs = classify_mcq_suitability(mcqs, question_col="question")

    # Step 2: convert
    converted = convert_mcqs_to_osq(mcqs, only_if_suitable=True)

    print(converted[["id", "osq_suitable", "osq_prompt", "expected_answer", "difficulty"]].to_string())


   id osq_suitable                                                                                                      osq_prompt                                                                                   expected_answer difficulty
0  Q1          Yes  Describe the lifecycle model that emphasizes iterative risk reduction through continuous stakeholder feedback.  The Spiral model emphasizes iterative cycles with risk-driven prototyping and stakeholder input.     medium
1  Q2          Yes                                    What is the formula for calculating the Shannon capacity of an AWGN channel?                                                                             C = B * log2(1 + SNR)     medium


# Adapting for SysEngBench

In [5]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
# Now you can read vars
api_key = os.getenv("OPENROUTER_API_KEY")
print("Loaded:", bool(api_key))

Loaded: True


In [7]:
# === Enhanced MCQ → OSQ Converter (preserving original schema) ===
# Prereqs: pip install openai>=1.0.0 pandas
# Make sure OPENAI_API_KEY is set in your environment before running.



import os, re, json
import pandas as pd
from typing import List, Dict, Any
from openai import OpenAI
from pathlib import Path

# -----------------------------
# Config
# -----------------------------
CSV_IN = "C:\\Users\\rabel\\Desktop\\dissertation\\src\\phase1_prep\\sysengbench.csv"
# CSV_OUT = "C:\\Users\\rabel\\Desktop\\dissertation\\src\\phase2_conversion\\sysengbench_converted.csv"
# JSONL_OUT = "C:\\Users\\rabel\\Desktop\\dissertation\\src\\phase2_conversion\\sysengbench_converted.jsonl"
NOTEBOOK_DIR = Path.cwd()  # folder of the running notebook
CSV_IN_PATH = Path(CSV_IN)
CSV_OUT   = NOTEBOOK_DIR / f"{CSV_IN_PATH.stem}_osq.csv"
JSONL_OUT = NOTEBOOK_DIR / f"{CSV_IN_PATH.stem}_osq.jsonl"


# MODEL = "gpt-4o-mini" # OpenAI configuration
MODEL = "openai/gpt-5" # OpenRouter configuration
# MODEL = "openai/gpt-5-mini" # OpenRouter configuration
SAMPLE_N = 5  # set >0 to test on small sample
CONFIDENCE_THRESHOLD = 7  # threshold for OSQ suitability (1-10 scale)

# -----------------------------
# Client
# -----------------------------
# client = OpenAI()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)


# -----------------------------
# Enhanced Prompts
# -----------------------------
CLASSIFIER_PROMPT = """You are an educational-assessment expert. Score how suitable a multiple-choice question (MCQ) is for conversion into an open-ended short-answer question (OSQ) **without losing validity**.

SCORE FROM 1–10 (single number). Use these five dimensions to determine the score, but DO NOT output pass/fail or the dimensions themselves:
1) Assessable without showing options.
2) Canonical core answer exists (definition, short derivation, numeric+units, or specific rationale).
3) Removing options won’t add major ambiguity or change the construct being measured.
4) Scope is constrainable in ≤2 sentences; expected answer fits in 1–6 sentences or a numeric expression.
5) Intent and semantics are preserved after conversion.

Rubric anchors (map your final score to the **highest level fully satisfied**):
- 9–10 EXCELLENT: Clear canonical answer; minimal ambiguity; tight scope; construct preserved.
- 7–8 GOOD: Conceptual/application/compare–contrast with crisp boundaries; minor rewording OK.
- 5–6 MARGINAL: Convertible with notable rewording/context; some dependence on options.
- 3–4 POOR: Heavy option dependence or ambiguity; conversion risks construct drift.
- 1–2 UNSUITABLE: No standalone assessment possible; relies on elimination/comparison of options.

EVALUATION QUESTION:
{question}

Return output ONLY as valid JSON (no extra text):
{
  "suitability_score": <integer 1-10>,
  "justification": "<2–3 sentences referencing the rubric anchors and the most decisive dimensions>"
}
"""



CONVERTER_PROMPT = """You convert exactly one multiple-choice question (MCQ) into a precise open-ended short-answer question (OSQ), keeping the construct and difficulty comparable. Assume a prior filter has confirmed this MCQ is suitable for conversion.

Conversion rules (apply all):
1) Remove all multiple-choice artifacts (“which of the following…”, letters A–D, option references). Do not mention options.
2) Preserve the original intent/construct. If the MCQ targeted recall, keep it recall; if numeric or procedural, keep it numeric/procedural. Do not broaden or narrow scope.
3) Keep difficulty approximately the same: if the original expects a fact, definition, derivation, numeric value (with units), or short rationale, require that explicitly.
4) For computations: state givens/assumptions the respondent must use; require units; specify the numeric format (e.g., significant figures or decimal places) and any constants.
5) Produce a canonical expected answer that a knowledgeable grader can match unambiguously. If multiple phrasings or equivalent forms are acceptable, list them in the rubric.
6) Create a detailed rubric that a second grader could apply consistently: point values, criteria for full/partial/no credit, and common acceptable variants and common errors.
7) Identify the Bloom’s taxonomy level that best matches the **intended cognitive action** of the converted OSQ (not the original MCQ stem wording) and justify briefly.

Formatting constraints:
- Be concise, precise, and neutral in tone.
- Do NOT reveal solutions inside the OSQ prompt.
- If numeric: specify required units and rounding (e.g., “Report to 2 decimal places with SI units”).
- If definitional/short rationale: set a length expectation (e.g., “Answer in 1–3 sentences”).
- No extraneous fields or text beyond the JSON. Escape quotes as needed.

Return STRICT JSON with fields:
{
  "osq_prompt": "<final open-ended question users will see>",
  "expected_answer": "<concise canonical answer; if numeric, include units and rounding target>",
  "rubric": {
    "full_credit": "<criteria and points; list acceptable synonyms/representations if any>",
    "partial_credit": "<criteria and points for partially correct work; note common correct elements and typical mistakes>",
    "no_credit": "<criteria for no credit; note disqualifying errors or missing elements>"
  },
  "blooms_level": "<one of: Remember, Understand, Apply, Analyze, Evaluate, Create>",
  "blooms_justification": "<1–2 sentences: identify the cognitive operation elicited by the OSQ prompt and why>"
}

"""

# -----------------------------
# API Helper
# -----------------------------
def chat_json(prompt: str, model: str = MODEL, temperature: float = 0.0, max_tokens: int = 800) -> Dict[str, Any]:
    """Call Chat Completions and parse JSON response robustly."""
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    text = resp.choices[0].message.content.strip()
    
    # Clean up markdown code blocks if present
    if text.startswith("```"):
        text = text.strip("`")
        idx = text.find("{")
        if idx >= 0:
            text = text[idx:]
    
    try:
        return json.loads(text)
    except Exception:
        # Try to extract JSON from text
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            try:
                return json.loads(text[start:end+1])
            except:
                pass
        raise ValueError(f"Model did not return valid JSON:\n{text}")

# -----------------------------
# Data Processing Functions
# -----------------------------
def parse_choices_list(choices_str) -> List[str]:
    """Parse choices from string format to list."""
    if pd.isna(choices_str) or choices_str == "":
        return []
    
    # Remove brackets and split by newlines or semicolons
    choices_str = str(choices_str).strip()
    if choices_str.startswith('[') and choices_str.endswith(']'):
        choices_str = choices_str[1:-1]
    
    # Split and clean
    choices = [choice.strip().strip('"\'') for choice in choices_str.split('\n') if choice.strip()]
    return [choice for choice in choices if choice and choice != '']

def preserve_original_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure the DataFrame preserves original HuggingFace schema structure."""
    # Expected columns from the original dataset
    original_columns = [
        'id', 'topics', 'question', 'choice_a', 'choice_b', 'choice_c', 'choice_d',
        'correct_choice', 'explanation'
    ]
    
    # Create choices list from individual choice columns
    if 'choices' not in df.columns:
        def create_choices_list(row):
            choices = []
            for col in ['choice_a', 'choice_b', 'choice_c', 'choice_d']:
                if col in row and pd.notna(row[col]) and str(row[col]).strip():
                    choices.append(str(row[col]).strip())
            return choices
        
        df['choices'] = df.apply(create_choices_list, axis=1)
    
    # Ensure all original columns exist
    for col in original_columns:
        if col not in df.columns:
            df[col] = ""
    
    return df

def classify_mcq_suitability(df: pd.DataFrame, 
                           confidence_threshold: int = CONFIDENCE_THRESHOLD,
                           model: str = MODEL) -> pd.DataFrame:
    """Classify MCQs for OSQ suitability with confidence scoring."""
    results = []
    
    for _, row in df.iterrows():
        question = row.get('question', '')
        prompt = f"{CLASSIFIER_PROMPT}\n\nMCQ to evaluate:\n{question}"
        
        try:
            response = chat_json(prompt, model=model, temperature=0.0, max_tokens=300)
            decision = response.get('decision', 'No')
            confidence = response.get('confidence', 0)
            justification = response.get('justification', 'No justification provided')
            
            # Apply threshold
            osq_suitable = "Yes" if (decision == "Yes" and confidence >= confidence_threshold) else "No"
            
            results.append({
                'osq_suitable': osq_suitable,
                'confidence_level': confidence,
                'classification_justification': justification
            })
            
        except Exception as e:
            print(f"Error classifying question {row.get('id', 'unknown')}: {e}")
            results.append({
                'osq_suitable': 'No',
                'confidence_level': 0,
                'classification_justification': f'Error in classification: {str(e)}'
            })
    
    # Add results to dataframe
    for i, result in enumerate(results):
        for key, value in result.items():
            df.iloc[i, df.columns.get_loc(key) if key in df.columns else len(df.columns)] = value
            if key not in df.columns:
                df[key] = ""
                df.iloc[i, df.columns.get_loc(key)] = value
    
    return df

def convert_mcq_to_osq(row: pd.Series, model: str = MODEL) -> Dict[str, Any]:
    """Convert a single MCQ row to OSQ format with enhanced rubric and Bloom's analysis."""
    question = row.get('question', '')
    choices = row.get('choices', [])
    correct_choice = row.get('correct_choice', '')
    explanation = row.get('explanation', '')
    
    prompt = f"""{CONVERTER_PROMPT}

Input MCQ:
Question: {question}
Choices: {json.dumps(choices, ensure_ascii=False)}
Correct Choice: {correct_choice}
Explanation: {explanation}
"""
    
    try:
        response = chat_json(prompt, model=model, temperature=0.2, max_tokens=1000)
        return {
            'osq_prompt': response.get('osq_prompt', ''),
            'expected_answer': response.get('expected_answer', ''),
            'full_credit_criteria': response.get('rubric', {}).get('full_credit', ''),
            'partial_credit_criteria': response.get('rubric', {}).get('partial_credit', ''),
            'no_credit_criteria': response.get('rubric', {}).get('no_credit', ''),
            'blooms_level': response.get('blooms_level', ''),
            'blooms_justification': response.get('blooms_justification', '')
        }
    except Exception as e:
        print(f"Error converting question {row.get('id', 'unknown')}: {e}")
        return {
            'osq_prompt': '',
            'expected_answer': '',
            'full_credit_criteria': '',
            'partial_credit_criteria': '',
            'no_credit_criteria': '',
            'blooms_level': '',
            'blooms_justification': f'Error in conversion: {str(e)}'
        }

def process_conversions(df: pd.DataFrame, model: str = MODEL) -> pd.DataFrame:
    """Process OSQ conversions for suitable MCQs."""
    conversion_results = []
    
    for _, row in df.iterrows():
        if row.get('osq_suitable', 'No') == 'Yes':
            result = convert_mcq_to_osq(row, model=model)
        else:
            # Empty results for unsuitable questions
            result = {
                'osq_prompt': '',
                'expected_answer': '',
                'full_credit_criteria': '',
                'partial_credit_criteria': '',
                'no_credit_criteria': '',
                'blooms_level': '',
                'blooms_justification': ''
            }
        conversion_results.append(result)
    
    # Add conversion results to dataframe
    for i, result in enumerate(conversion_results):
        for key, value in result.items():
            if key not in df.columns:
                df[key] = ""
            df.iloc[i, df.columns.get_loc(key)] = value
    
    return df

# -----------------------------
# Main Execution
# -----------------------------
def main():
    print("Loading and processing MCQ dataset...")
    
    # Load data
    df_raw = pd.read_csv(CSV_IN)
    print(f"Loaded {len(df_raw)} questions from {CSV_IN}")
    
    # Preserve original schema
    df = preserve_original_schema(df_raw.copy())
    
    # Sample if specified
    if SAMPLE_N and SAMPLE_N > 0:
        df = df.sample(n=min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)
        print(f"Using sample of {len(df)} questions for testing")
    
    print("Classifying MCQs for OSQ suitability...")
    df = classify_mcq_suitability(df, confidence_threshold=CONFIDENCE_THRESHOLD)
    
    suitable_count = len(df[df['osq_suitable'] == 'Yes'])
    print(f"Found {suitable_count} questions suitable for OSQ conversion")
    
    print("Converting suitable MCQs to OSQ format...")
    df = process_conversions(df)
    
    # Define final column order
    original_cols = ['id', 'topics', 'question', 'choice_a', 'choice_b', 'choice_c', 'choice_d', 
                    'correct_choice', 'explanation', 'choices']
    classification_cols = ['osq_suitable', 'confidence_level', 'classification_justification']
    conversion_cols = ['osq_prompt', 'expected_answer', 'full_credit_criteria', 
                      'partial_credit_criteria', 'no_credit_criteria', 'blooms_level', 
                      'blooms_justification']
    
    final_columns = original_cols + classification_cols + conversion_cols
    
    # Ensure all columns exist and reorder
    for col in final_columns:
        if col not in df.columns:
            df[col] = ""
    
    df_final = df[final_columns]
    
    # Save outputs
    print(f"Saving results to {CSV_OUT} and {JSONL_OUT}")
    df_final.to_csv(CSV_OUT, index=False)
    
    # Save JSONL
    with open(JSONL_OUT, "w", encoding="utf-8") as f:
        for _, row in df_final.iterrows():
            record = row.to_dict()
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    
    # Display summary
    print(f"\n=== PROCESSING SUMMARY ===")
    print(f"Total questions processed: {len(df_final)}")
    print(f"Questions suitable for OSQ: {suitable_count}")
    print(f"Conversion success rate: {suitable_count/len(df_final)*100:.1f}%")
    print(f"Average confidence level: {df_final['confidence_level'].mean():.1f}")
    
    print(f"\n=== SAMPLE RESULTS ===")
    sample_cols = ['id', 'osq_suitable', 'confidence_level', 'osq_prompt', 'blooms_level']
    print(df_final[sample_cols].head(3).to_string(index=False))
    
    print(f"\nFiles saved:")
    print(f"  CSV: {CSV_OUT}")
    print(f"  JSONL: {JSONL_OUT}")

if __name__ == "__main__":
    main()

Loading and processing MCQ dataset...
Loaded 1144 questions from C:\Users\rabel\Desktop\dissertation\src\phase1_prep\sysengbench.csv
Using sample of 5 questions for testing
Classifying MCQs for OSQ suitability...
Error classifying question : Model did not return valid JSON:

Error classifying question : Model did not return valid JSON:

Error classifying question : Model did not return valid JSON:

Error classifying question : Model did not return valid JSON:

Error classifying question : Model did not return valid JSON:



IndexError: iloc cannot enlarge its target object

# Simplifying to remove errors

In [8]:
# %pip install python-dotenv openai pandas

import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# Load .env from nearest parent
load_dotenv(find_dotenv())
print("OPENROUTER_API_KEY present:", bool(os.getenv("OPENROUTER_API_KEY")))


OPENROUTER_API_KEY present: True


In [9]:
import os, re, json, time, logging
from pathlib import Path
from typing import List, Dict, Any

import pandas as pd
from openai import OpenAI

# ---------- logging ----------
LOG_LEVEL = logging.DEBUG  # DEBUG for verbose; switch to INFO to quiet down
logging.basicConfig(
    level=LOG_LEVEL,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("mcq2osq")

# ---------- paths & params ----------
CSV_IN = r"C:\Users\rabel\Desktop\dissertation\src\phase1_prep\sysengbench.csv"
NOTEBOOK_DIR = Path.cwd()
CSV_IN_PATH  = Path(CSV_IN)
CSV_OUT      = NOTEBOOK_DIR / f"{CSV_IN_PATH.stem}_osq.csv"
JSONL_OUT    = NOTEBOOK_DIR / f"{CSV_IN_PATH.stem}_osq.jsonl"
RAW_LOG_DIR  = NOTEBOOK_DIR / "model_raw_logs"   # stores raw model outputs per row
RAW_LOG_DIR.mkdir(exist_ok=True)

MODEL = "openai/gpt-5"   # via OpenRouter
SAMPLE_N = 5             # small sample for test runs; set 0 to process all
CONFIDENCE_THRESHOLD = 7 # suitability threshold (1-10)

# ---------- client ----------
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)
log.info("Client ready")


22:57:28 | INFO | mcq2osq | Client ready


In [ ]:
CLASSIFIER_PROMPT = """You are an educational-assessment expert. Score how suitable a multiple-choice question (MCQ) is for conversion into an open-ended short-answer question (OSQ) **without losing validity**.

SCORE FROM 1–10 (single number). Use these five dimensions to determine the score, but DO NOT output pass/fail or the dimensions themselves:
1) Assessable without showing options.
2) Canonical core answer exists (definition, short derivation, numeric+units, or specific rationale).
3) Removing options won’t add major ambiguity or change the construct being measured.
4) Scope is constrainable in ≤2 sentences; expected answer fits in 1–6 sentences or a numeric expression.
5) Intent and semantics are preserved after conversion.

Rubric anchors (map your final score to the **highest level fully satisfied**):
- 9–10 EXCELLENT: Clear canonical answer; minimal ambiguity; tight scope; construct preserved.
- 7–8 GOOD: Conceptual/application/compare–contrast with crisp boundaries; minor rewording OK.
- 5–6 MARGINAL: Convertible with notable rewording/context; some dependence on options.
- 3–4 POOR: Heavy option dependence or ambiguity; conversion risks construct drift.
- 1–2 UNSUITABLE: No standalone assessment possible; relies on elimination/comparison of options.

EVALUATION QUESTION:
{question}

Return output ONLY as valid JSON (no extra text):
{{
  "suitability_score": <integer 1-10>,
  "justification": "<2–3 sentences referencing the rubric anchors and the most decisive dimensions>"
}}

DO NOT add any text outside the JSON object—no explanations, no markdown code fences.
"""

CONVERTER_PROMPT = """You convert exactly one multiple-choice question (MCQ) into a precise open-ended short-answer question (OSQ), keeping the construct and difficulty comparable. Assume a prior filter has confirmed this MCQ is suitable for conversion.

Conversion rules (apply all):
1) Remove all multiple-choice artifacts (“which of the following…”, letters A–D, option references). Do not mention options.
2) Preserve the original intent/construct. If the MCQ targeted recall, keep it recall; if numeric or procedural, keep it numeric/procedural. Do not broaden or narrow scope.
3) Keep difficulty approximately the same: if the original expects a fact, definition, derivation, numeric value (with units), or short rationale, require that explicitly.
4) For computations: state givens/assumptions the respondent must use; require units; specify the numeric format (e.g., significant figures or decimal places) and any constants.
5) Produce a canonical expected answer that a knowledgeable grader can match unambiguously. If multiple phrasings or equivalent forms are acceptable, list them in the rubric.
6) Create a detailed rubric that a second grader could apply consistently: point values, criteria for full/partial/no credit, and common acceptable variants and common errors.
7) Identify the Bloom’s taxonomy level that best matches the **intended cognitive action** of the converted OSQ (not the original MCQ stem wording) and justify briefly.

Formatting constraints:
- Be concise, precise, and neutral in tone.
- Do NOT reveal solutions inside the OSQ prompt.
- If numeric: specify required units and rounding (e.g., “Report to 2 decimal places with SI units”).
- If definitional/short rationale: set a length expectation (e.g., “Answer in 1–3 sentences”).
- No extraneous fields or text beyond the JSON. Escape quotes as needed.

Return STRICT JSON with fields:
{{
  "osq_prompt": "<final open-ended question users will see>",
  "expected_answer": "<concise canonical answer; if numeric, include units and rounding target>",
  "rubric": {
    "full_credit": "<criteria and points; list acceptable synonyms/representations if any>",
    "partial_credit": "<criteria and points for partially correct work; note common correct elements and typical mistakes>",
    "no_credit": "<criteria for no credit; note disqualifying errors or missing elements>"
  },
  "blooms_level": "<one of: Remember, Understand, Apply, Analyze, Evaluate, Create>",
  "blooms_justification": "<1–2 sentences: identify the cognitive operation elicited by the OSQ prompt and why>"
}}

DO NOT add any text outside the JSON object—no explanations, no markdown code fences.
"""
log.debug("Prompts loaded")


22:57:28 | DEBUG | mcq2osq | Prompts loaded


In [11]:
def _clean_to_json_block(text: str) -> str:
    """Strip fences; extract first {...} block if extra text leaked in."""
    # Remove markdown fences (```json ... ```)
    text = re.sub(r"^```(?:json)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text)

    if text.startswith("{") and text.endswith("}"):
        return text

    m = re.search(r"\{.*\}", text, re.S)
    return m.group(0) if m else text  # may still fail json.loads


def chat_json(prompt: str, *, model: str, temperature: float, max_tokens: int,
              retries: int = 1, log_prefix: str = "", save_raw_path: Path | None = None) -> Dict[str, Any]:
    """
    Call the chat model and parse JSON robustly.
    - Logs raw response (truncated) + cleaned block.
    - Optionally saves the full raw text to disk (for postmortem).
    """
    for attempt in range(retries + 1):
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
        )
        text = resp.choices[0].message.content or ""
        if save_raw_path:
            save_raw_path.write_text(text, encoding="utf-8")

        short = text[:400].replace("\n", "\\n")
        log.debug(f"{log_prefix} raw[{len(text)}]: {short}{'…' if len(text)>400 else ''}")

        cleaned = _clean_to_json_block(text)
        cshort  = cleaned[:400].replace("\n", "\\n")
        log.debug(f"{log_prefix} cleaned[{len(cleaned)}]: {cshort}{'…' if len(cleaned)>400 else ''}")

        try:
            out = json.loads(cleaned)
            return out
        except Exception as e:
            log.warning(f"{log_prefix} JSON parse failed (attempt {attempt+1}/{retries+1}): {e}")
            if attempt < retries:
                prompt += "\n\nReminder: Return ONLY valid JSON with no extra text."
                time.sleep(0.4)  # tiny backoff
            else:
                raise ValueError(f"{log_prefix} Model did not return valid JSON:\n{cleaned}") from e


In [12]:
def parse_choices_list(raw) -> List[str]:
    """Convert a cell (list-like or string) to a clean list of choices."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)) or (isinstance(raw, str) and not raw.strip()):
        return []
    text = str(raw).strip(" []\n\r\t")
    return [c.strip(' "\'') for c in re.split(r'[\n;]+', text) if c.strip()]

def preserve_original_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure expected columns exist and build a 'choices' list from choice_a..d if missing.
    """
    needed = [
        'id','topics','question','choice_a','choice_b',
        'choice_c','choice_d','correct_choice','explanation'
    ]
    for col in needed:
        if col not in df.columns:
            df[col] = ""

    if 'choices' not in df.columns:
        df['choices'] = df.apply(
            lambda r: [str(r[c]).strip() for c in ['choice_a','choice_b','choice_c','choice_d']
                       if c in r and pd.notna(r[c]) and str(r[c]).strip()],
            axis=1
        )
    return df


In [13]:
def classify_mcq_suitability(df: pd.DataFrame,
                             confidence_threshold: int = CONFIDENCE_THRESHOLD,
                             model: str = MODEL) -> pd.DataFrame:
    """
    Score each MCQ (1–10). Adds:
      - suitability_score (int)
      - confidence_level (alias of suitability_score for backward compatibility)
      - classification_justification (str)
      - osq_suitable ('Yes'/'No' using threshold)
    """
    scores, justs, flags = [], [], []

    for idx, row in df.iterrows():
        qid = row.get('id', idx)
        question = row.get('question', '')
        prompt = CLASSIFIER_PROMPT.format(question=question)
        save_path = RAW_LOG_DIR / f"classifier_raw_{qid}.txt"

        log.info(f"[CLS] id={qid} | q.len={len(str(question))}")
        try:
            resp = chat_json(
                prompt,
                model=model,
                temperature=0.0,
                max_tokens=600,
                retries=1,
                log_prefix=f"[CLS id={qid}]",
                save_raw_path=save_path
            )
            score = int(resp.get('suitability_score', 0))
            just  = resp.get('justification', '')
            log.debug(f"[CLS id={qid}] score={score} | just={just}")
        except Exception as e:
            log.error(f"[CLS id={qid}] failed: {e}")
            score, just = 0, f"Error: {e}"

        scores.append(score)
        justs.append(just)
        flags.append('Yes' if score >= confidence_threshold else 'No')

    out = df.copy()
    out['suitability_score'] = scores
    out['confidence_level'] = scores  # keep existing downstream summaries working
    out['classification_justification'] = justs
    out['osq_suitable'] = flags
    return out


In [14]:
def convert_mcq_to_osq(row: pd.Series, model: str = MODEL) -> Dict[str, Any]:
    """
    Convert a single MCQ to OSQ with rubric and Bloom's analysis.
    Logs raw response and returns parsed fields.
    """
    qid = row.get('id', 'unknown')
    prompt = f"""{CONVERTER_PROMPT}

Input MCQ:
Question: {row.get('question', '')}
Choices: {json.dumps(row.get('choices', []), ensure_ascii=False)}
Correct Choice: {row.get('correct_choice', '')}
Explanation: {row.get('explanation', '')}
"""
    log.info(f"[CVT] id={qid} | converting…")
    save_path = RAW_LOG_DIR / f"converter_raw_{qid}.txt"
    try:
        resp = chat_json(
            prompt,
            model=model,
            temperature=0.2,
            max_tokens=1100,
            retries=1,
            log_prefix=f"[CVT id={qid}]",
            save_raw_path=save_path
        )
        r = resp.get('rubric', {}) or {}
        out = {
            'osq_prompt': resp.get('osq_prompt', '').strip(),
            'expected_answer': resp.get('expected_answer', '').strip(),
            'full_credit_criteria': r.get('full_credit', '').strip(),
            'partial_credit_criteria': r.get('partial_credit', '').strip(),
            'no_credit_criteria': r.get('no_credit', '').strip(),
            'blooms_level': resp.get('blooms_level', '').strip(),
            'blooms_justification': resp.get('blooms_justification', '').strip(),
        }
        log.debug(f"[CVT id={qid}] done | has_prompt={bool(out['osq_prompt'])}")
        return out
    except Exception as e:
        log.error(f"[CVT id={qid}] failed: {e}")
        return {
            'osq_prompt':'','expected_answer':'',
            'full_credit_criteria':'','partial_credit_criteria':'','no_credit_criteria':'',
            'blooms_level':'','blooms_justification':f'Error: {e}'
        }


In [15]:
def process_conversions(df: pd.DataFrame, model: str = MODEL) -> pd.DataFrame:
    """
    Apply convert_mcq_to_osq only to rows with osq_suitable == 'Yes'.
    """
    results: List[Dict[str, Any]] = []
    total = len(df)
    for i, row in df.iterrows():
        if row.get('osq_suitable') == 'Yes':
            res = convert_mcq_to_osq(row, model=model)
        else:
            res = {
                'osq_prompt':'','expected_answer':'',
                'full_credit_criteria':'','partial_credit_criteria':'','no_credit_criteria':'',
                'blooms_level':'','blooms_justification':''
            }
        results.append(res)
        if (i+1) % 5 == 0 or i == total - 1:
            log.info(f"[CVT] progress: {i+1}/{total}")

    add = pd.DataFrame(results)
    return pd.concat([df.reset_index(drop=True), add], axis=1)


In [16]:
print("SAMPLE_N =", SAMPLE_N)

SAMPLE_N = 5


In [17]:
def main():
    log.info("Loading CSV…")
    df_raw = pd.read_csv(CSV_IN)
    log.info(f"Loaded {len(df_raw)} rows from {CSV_IN}")

    df = preserve_original_schema(df_raw.copy())

    if SAMPLE_N and SAMPLE_N > 0:
        df = df.sample(n=min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)
        log.info(f"Sampled {len(df)} rows (SAMPLE_N={SAMPLE_N})")

    log.info("Classifying OSQ suitability…")
    df = classify_mcq_suitability(df, confidence_threshold=CONFIDENCE_THRESHOLD, model=MODEL)
    log.info(f"Suitable count (≥{CONFIDENCE_THRESHOLD}): {(df['osq_suitable']=='Yes').sum()} / {len(df)}")

    log.info("Converting suitable MCQs → OSQs…")
    df = process_conversions(df, model=MODEL)

    # final col order
    original_cols = ['id','topics','question','choice_a','choice_b','choice_c','choice_d',
                     'correct_choice','explanation','choices']
    classification_cols = ['osq_suitable','suitability_score','confidence_level','classification_justification']
    conversion_cols = ['osq_prompt','expected_answer','full_credit_criteria',
                       'partial_credit_criteria','no_credit_criteria','blooms_level','blooms_justification']

    for col in original_cols + classification_cols + conversion_cols:
        if col not in df.columns: df[col] = ""

    df_final = df[original_cols + classification_cols + conversion_cols]

    log.info(f"Saving CSV → {CSV_OUT}")
    df_final.to_csv(CSV_OUT, index=False)

    log.info(f"Saving JSONL → {JSONL_OUT}")
    with open(JSONL_OUT, "w", encoding="utf-8") as f:
        for _, row in df_final.iterrows():
            f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

    # summary
    suitable_count = (df_final['osq_suitable']=='Yes').sum()
    log.info("=== SUMMARY ===")
    log.info(f"Total processed: {len(df_final)}")
    log.info(f"Suitable: {suitable_count} ({suitable_count/len(df_final)*100:.1f}%)")
    log.info(f"Average suitability score: {df_final['suitability_score'].astype(float).mean():.2f}")

    # peek
    peek_cols = ['id','osq_suitable','suitability_score','osq_prompt','blooms_level']
    log.info("\n" + df_final[peek_cols].head(3).to_string(index=False))

    print("\nFiles saved:")
    print(f"  CSV:   {CSV_OUT}")
    print(f"  JSONL: {JSONL_OUT}")

if __name__ == "__main__":
    main()


22:58:09 | INFO | mcq2osq | Loading CSV…
22:58:09 | INFO | mcq2osq | Loaded 1144 rows from C:\Users\rabel\Desktop\dissertation\src\phase1_prep\sysengbench.csv
22:58:09 | INFO | mcq2osq | Sampled 5 rows (SAMPLE_N=5)
22:58:09 | INFO | mcq2osq | Classifying OSQ suitability…


KeyError: '\n  "suitability_score"'